# 사전구축 라이브러리 — `langgraph-swarm`

**Swarm** 은 에이전트들이 **동적으로 서로에게 제어권을 넘기는(hand-off)** 멀티에이전트 구조다. Network 와 비슷하지만, **"지금 활성화된 에이전트"** 개념이 있어 대화가 한 에이전트에 머물다가 필요할 때 다른 에이전트로 넘어가고, 그 상태가 유지된다.

`langgraph-swarm` 의 두 핵심:
- **`create_handoff_tool(agent_name=...)`**: "이 에이전트로 넘겨라" 라는 핸드오프 도구
- **`create_swarm([...], default_active_agent=...)`**: 스웜 그래프 생성 (시작 에이전트 지정)

예제: 시장조사 전문가 ⇄ 마케팅 전문가.

> `pip/uv install langgraph-swarm`. `OPENAI_API_KEY`, `TAVILY_API_KEY` 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
for k in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(k), f"{k} 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 1. 최소 예제 — Alice ⇄ Bob

Alice(덧셈 전문) 와 Bob(해적 말투) 가 서로 핸드오프한다. [prebuilt] 각 에이전트는 `create_handoff_tool(agent_name=상대)` 를 도구로 가져, 필요하면 그 도구를 호출해 상대에게 넘긴다. 상태는 checkpointer 로 유지(어느 에이전트가 활성인지 기억).

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent
from langgraph_swarm import create_handoff_tool, create_swarm

model = ChatOpenAI(model="gpt-4o")

def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

alice = create_react_agent(
    model,
    [add, create_handoff_tool(agent_name="Bob")],
    prompt="You are Alice, an addition expert.",
    name="Alice",
)
bob = create_react_agent(
    model,
    [create_handoff_tool(agent_name="Alice", description="Transfer to Alice, she can help with math")],
    prompt="You are Bob, you speak like a pirate.",
    name="Bob",
)

workflow = create_swarm([alice, bob], default_active_agent="Alice")
app = workflow.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# 1턴: Bob 에게 넘겨달라 → Alice 가 Bob 으로 핸드오프
turn_1 = app.invoke({"messages": [{"role": "user", "content": "i'd like to speak to Bob"}]}, config)
turn_1["messages"][-1].pretty_print()

# 2턴: 수학 질문 → Bob 이 Alice 로 다시 핸드오프 (활성 에이전트가 이어짐)
turn_2 = app.invoke({"messages": [{"role": "user", "content": "what's 5 + 7?"}]}, config)
turn_2["messages"][-1].pretty_print()

## 2. 시장조사 ⇄ 마케팅 Swarm

### 시장조사 에이전트
웹검색 + 결과를 표로 변환하는 도구. 마케팅이 필요하면 Marketing 으로 핸드오프.
[basics] `convert_to_table` 은 도구 안에서 LLM 체인을 호출하는 형태(03 튜터의 `return_diagnosis` 와 같은 발상).

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated

tavily_tool = TavilySearchResults(max_results=5)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

table_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You convert unstructured text into a well-organized Markdown table. "
     "Infer appropriate columns, extract data, return only the Markdown table "
     "(no explanation, no code fences). Include a header row."),
    ("user", "Input text: {raw_text}"),
])
table_chain = table_prompt | llm

@tool
def convert_to_table(
    raw_text: Annotated[str, "Unstructured text to transform into a table"]
) -> str:
    """Use an LLM to infer columns and convert text into a Markdown table."""
    return table_chain.invoke({"raw_text": raw_text}).content

In [ ]:
from langgraph_swarm import create_handoff_tool, create_swarm

model = ChatOpenAI(model="gpt-4o")

searching_agent = create_react_agent(
    model,
    [tavily_tool, convert_to_table,
     create_handoff_tool(agent_name="Marketing",
        description="If the user needs marketing strategy, transfer to the Marketing Agent.")],
    prompt="You conduct market research. Search the web for accurate, up-to-date info on "
           "markets, industries, competitors, or consumer trends.",
    name="Searching",
)

### 마케팅 에이전트
마케팅 문구 생성 / 셀링포인트 추출 / 퍼널 단계 분석 도구. 시장조사가 필요하면 Searching 으로 핸드오프.

In [ ]:
from typing import List, Dict
import json

@tool
def write_marketing_copy(
    keywords: Annotated[List[str], "Features to base the copy on (e.g. eco-friendly, AI-powered)."]
) -> List[str]:
    """Generate short, punchy marketing phrases (<15 words each) for product features."""
    prompt = (
        "Create several short, punchy marketing phrases capturing all these features. "
        "Natural, unique, catchy, under 15 words each:\n\n"
        + "\n".join(f"- {kw}" for kw in keywords)
    )
    resp = model.invoke(prompt)
    return [line.strip("-• ").strip() for line in resp.content.split("\n") if line.strip()]

@tool
def extract_selling_points(
    product_description: Annotated[str, "Description of the product or service."]
) -> List[str]:
    """Extract compelling, benefit-driven selling points from a product description."""
    prompt = (
        "You are a top-tier brand copywriter. Extract the strongest, most persuasive "
        "selling points from the description below. Focus on real benefits and what sets it apart.\n\n"
        f"Product Description:\n{product_description}"
    )
    resp = model.invoke(prompt)
    return [line.strip("-• ").strip() for line in resp.content.split("\n") if line.strip()]

@tool
def extract_marketing_funnel_stages(
    marketing_text: Annotated[str, "Marketing content to infer customer journey stages from."]
) -> Dict[str, str]:
    """Map content to funnel stages: Awareness, Consideration, Purchase, Retention, Advocacy."""
    prompt = (
        "You are a marketing strategist. Map the content below to funnel stages "
        "(Awareness, Consideration, Purchase, Retention, Advocacy). For each, extract the most "
        "relevant message/CTA. Return JSON with those keys (no code fences); empty string if absent.\n\n"
        f"Marketing Text:\n{marketing_text}"
    )
    resp = model.invoke(prompt)
    try:
        return json.loads(resp.content)
    except Exception:
        return {}

In [ ]:
marketing_agent = create_react_agent(
    model,
    [write_marketing_copy, extract_selling_points, extract_marketing_funnel_stages,
     create_handoff_tool(agent_name="Searching",
        description="If the user needs market research, transfer to the Market Research Agent.")],
    prompt="You are the Marketing Agent. Help with marketing tasks: ad copy, selling points, "
           "and marketing strategy analysis.",
    name="Marketing",
)

## 3. Swarm 조립
[prebuilt] 두 에이전트를 `create_swarm` 으로 묶고 시작 에이전트를 지정. checkpointer 로 활성 에이전트 상태 유지.

In [ ]:
checkpointer = InMemorySaver()
graph = create_swarm(
    [searching_agent, marketing_agent],
    default_active_agent="Searching",
).compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트
디카페인 음료 마케팅 요청 → (필요시 조사 ⇄ 마케팅 핸드오프) → 전략 산출.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

turn = graph.invoke(
    {"messages": [{"role": "user", "content":
        "새로운 디카페인 음료의 마케팅 전략을 세워줘. 말차 맛이고 비건 인증을 받았어. 한국어로 작성해줘."}]},
    config,
)
for msg in turn["messages"]:
    msg.pretty_print()

## 정리

- **Swarm** = 에이전트끼리 동적으로 제어권을 넘기되, **활성 에이전트** 상태가 유지되는 구조
- **`create_handoff_tool(agent_name=...)`**: 핸드오프를 도구로 — LLM 이 도구 호출로 넘김
- **`create_swarm([...], default_active_agent=...)`**: 스웜 그래프 생성, checkpointer 로 활성 상태 기억
- Network(직접 핸드오프)의 prebuilt 버전에 가깝고, '대화가 누구에게 머무는지' 개념이 추가됨

| prebuilt | 대응 패턴 |
|---|---|
| `langgraph-supervisor` | Supervisor (중앙 관리자) |
| `langgraph-swarm` | Network/Hand-off (동적 제어권 이양) |

직접 구현으로 원리를 익히고, 표준 구조는 prebuilt 로 빠르게 — 둘을 상황에 맞게 쓰면 된다.